[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/075.%20The%20VRP%20with%20Pickup%20and%20Delivery%20%28VRPPD%29/P75-Tier-7_executed.ipynb)

---


# 75. The Vehicle Routing Problem with Pickup and Delivery (VRPPD)
## Tier 7: Human–AI Collaboration and Explainable Decisions

### Purpose and scope
This notebook demonstrates a human–AI decision-support layer for a small pickup-and-delivery setting. It builds explanations for heuristic decisions, lets a simulated human operator evaluate them, records feedback, and summarizes the resulting interactions.

The notebook does **not** compute an optimal VRPPD solution. Its route-assignment rule is a greedy score, and its route-improvement step is a 2-opt local search. The simulation also includes priority decisions that are not part of the routing optimization model. Reported metrics describe this simulation and its scoring rules; they are not empirical evidence of real-world understanding, trust, or decision-quality improvements.

### Inputs and outputs
The instance contains pickup–delivery pairs, vehicles, a vehicle-capacity value, demands, and a symmetric distance matrix. The simulation produces decisions and explanations, simulated operator evaluations and interventions, trust histories, summary metrics, and plots. The fixed NumPy seed makes the generated instance reproducible. Python's `random` module is not seeded, so the sequence of simulated decisions and interventions can vary between runs.

### Modeling notes
For request \(r\), let \(p_r\) and \(d_r\) denote its pickup and delivery vertices. Its pickup demand is \(q_r > 0\), and its delivery demand is \(-q_r\). In a full pickup-and-delivery routing model, a vehicle's load changes as it visits these vertices, capacity must be respected throughout the route, and pickup must precede its paired delivery. This notebook stores these data but does not solve that full optimization model.

The route-assignment heuristic considers vehicles whose current load plus the request demand does not exceed capacity. For each eligible vehicle \(v\), it computes

$$
S_v = (c_{v,p_r} + c_{p_r,d_r})\left(1 + \frac{0.5 L_v}{Q}\right),
$$

where \(c_{i,j}\) is the distance from vertex \(i\) to vertex \(j\), \(L_v\) is the vehicle's current load, and \(Q\) is vehicle capacity. It selects a vehicle with the smallest score. This is a transparent greedy rule, not a global objective for the VRPPD.

For route improvement, 2-opt proposes reversing an internal segment of a route and accepts a proposal only when it shortens the route and preserves pickup-before-delivery precedence. The search examines pairs of segment boundaries; it is a local search and does not guarantee a globally shortest feasible route.

### What to examine
- How the score, capacity screen, and alternatives support an understandable assignment explanation.
- How the operator's evaluation changes simulated trust and whether the intervention rule triggers.
- How the analysis and plots summarize the simulation without treating simulated scores as validated human outcomes.
- Which operational constraints and learning mechanisms are outside this demonstration's scope.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

print("Libraries imported for the VRPPD human–AI decision-support demonstration.")

In [ ]:
@dataclass
class VRPPDInstance:
    """Data for a pickup-and-delivery demonstration instance."""

    num_pairs: int
    num_vehicles: int
    vehicle_capacity: int
    distances: np.ndarray
    demands: List[int]

    def __post_init__(self) -> None:
        self.num_vertices = 2 * self.num_pairs + 2
        self.depot_start = 0
        self.depot_end = 2 * self.num_pairs + 1
        self.pickups = list(range(1, self.num_pairs + 1))
        self.deliveries = list(range(self.num_pairs + 1, 2 * self.num_pairs + 1))
        self.all_vertices = list(range(self.num_vertices))
        self.vehicles = list(range(self.num_vehicles))

        if self.distances.shape != (self.num_vertices, self.num_vertices):
            raise ValueError("The distance matrix dimensions must match the vertex count.")
        if len(self.demands) != 2 * self.num_pairs:
            raise ValueError("Demands must contain one pickup and one delivery value per pair.")
        if not np.allclose(self.distances, self.distances.T):
            raise ValueError("This demonstration expects a symmetric distance matrix.")
        if not np.allclose(np.diag(self.distances), 0):
            raise ValueError("Distances from a vertex to itself must be zero.")


@dataclass
class DecisionExplanation:
    """Human-readable reasoning and factors associated with one AI decision."""

    decision_id: str
    decision_type: str
    reasoning: List[str]
    key_factors: Dict[str, float]
    alternatives: List[Dict[str, Any]]
    confidence: float
    timestamp: float


@dataclass
class HumanFeedback:
    """Simulated operator feedback on an AI decision."""

    feedback_id: str
    decision_id: str
    human_id: str
    rating: float
    comments: str
    corrections: Dict[str, Any]
    trust_impact: float
    timestamp: float


@dataclass
class TrustMetrics:
    """Trust values and satisfaction history for one simulated operator."""

    human_id: str
    trust_score: float
    competence_trust: float
    integrity_trust: float
    benevolence_trust: float
    interaction_count: int
    satisfaction_history: List[float]
    trust_history: List[float]


def create_partnership_instance() -> VRPPDInstance:
    """Create the fixed-size instance used by the demonstration."""
    num_pairs = 10
    num_vehicles = 6
    vehicle_capacity = 10

    np.random.seed(42)
    pickup_demands = np.random.randint(1, 5, num_pairs).tolist()
    delivery_demands = [-demand for demand in pickup_demands]
    demands = pickup_demands + delivery_demands

    num_vertices = 2 * num_pairs + 2
    distances = np.zeros((num_vertices, num_vertices))

    for i in range(num_vertices):
        for j in range(i + 1, num_vertices):
            base_distance = 8 + abs(i - j) * 1.5 + np.random.uniform(-1, 1)
            distances[i, j] = max(1, base_distance)
            distances[j, i] = distances[i, j]

    return VRPPDInstance(
        num_pairs=num_pairs,
        num_vehicles=num_vehicles,
        vehicle_capacity=vehicle_capacity,
        distances=distances,
        demands=demands,
    )


partnership_instance = create_partnership_instance()
print("VRPPD demonstration instance created:")
print(f"- Pickup–delivery pairs: {partnership_instance.num_pairs}")
print(f"- Vehicles: {partnership_instance.num_vehicles}")
print(f"- Vehicle capacity: {partnership_instance.vehicle_capacity}")
print(f"- Vertices, including start and end depots: {partnership_instance.num_vertices}")
print(f"- Sum of absolute pickup and delivery demands: {sum(abs(demand) for demand in partnership_instance.demands)}")

In [ ]:
class ExplainableAI:
    """Greedy VRPPD decision rules with explanations for their outputs."""

    def __init__(self, instance: VRPPDInstance):
        self.instance = instance
        self.decision_history: List[Dict[str, Any]] = []
        self.confidence_threshold = 0.7
        self.explanation_templates = {
            "route_assignment": [
                "Request {request_id} is assigned to vehicle {vehicle_id}.",
                "Current vehicle load: {current_load}/{capacity}.",
                "Distance from the current position to pickup: {distance:.1f} distance units.",
                "Pickup-and-delivery precedence is represented by the request pair.",
                "Other feasible vehicles considered: {alternatives_count}.",
            ],
            "priority_adjustment": [
                "Priority is being adjusted for request {request_id}.",
                "Original priority: {old_priority}.",
                "New priority: {new_priority}.",
                "Reason: {adjustment_reason}.",
                "Schedule implication: {schedule_impact}.",
            ],
            "route_optimization": [
                "A 2-opt search was applied to vehicle {vehicle_id}'s route.",
                "Current route length: {current_length:.1f} distance units.",
                "Distance reduction found: {potential_savings:.1f} distance units.",
                "The candidate must preserve pickup-before-delivery precedence.",
                "Search method: {method}.",
            ],
        }

    def calculate_route_distance(self, route: List[int]) -> float:
        """Sum the distances of consecutive vertices in a route."""
        return sum(
            self.instance.distances[left, right]
            for left, right in zip(route, route[1:])
        )

    def _precedence_is_respected(self, route: List[int]) -> bool:
        """Check that each represented pickup occurs before its delivery."""
        for pickup, delivery in zip(
            self.instance.pickups, self.instance.deliveries
        ):
            if pickup in route and delivery in route:
                if route.index(pickup) > route.index(delivery):
                    return False
        return True

    def make_decision_with_explanation(
        self, decision_type: str, context: Dict[str, Any]
    ) -> Tuple[Dict[str, Any], DecisionExplanation]:
        """Apply one named decision rule and record its explanation."""
        decision_id = f"decision_{len(self.decision_history)}_{time.time_ns()}"

        if decision_type == "route_assignment":
            decision, explanation = self._assign_route_with_explanation(
                context, decision_id
            )
        elif decision_type == "priority_adjustment":
            decision, explanation = self._adjust_priority_with_explanation(
                context, decision_id
            )
        elif decision_type == "route_optimization":
            decision, explanation = self._optimize_route_with_explanation(
                context, decision_id
            )
        else:
            decision = {"status": "unknown_decision_type"}
            explanation = DecisionExplanation(
                decision_id=decision_id,
                decision_type=decision_type,
                reasoning=["The requested decision type is not supported."],
                key_factors={},
                alternatives=[],
                confidence=0.0,
                timestamp=time.time(),
            )

        self.decision_history.append(
            {
                "decision_id": decision_id,
                "decision": decision,
                "explanation": explanation,
                "context": context,
            }
        )
        return decision, explanation

    def _assign_route_with_explanation(
        self, context: Dict[str, Any], decision_id: str
    ) -> Tuple[Dict[str, Any], DecisionExplanation]:
        request_id = context["request_id"]
        pickup_location = context["pickup_location"]
        delivery_location = context["delivery_location"]
        demand = context["demand"]
        vehicle_states = context["vehicle_states"]
        alternatives = []

        for vehicle_id, state in vehicle_states.items():
            if state["load"] + demand <= self.instance.vehicle_capacity:
                distance_to_pickup = self.instance.distances[
                    state["current_position"], pickup_location
                ]
                pickup_to_delivery = self.instance.distances[
                    pickup_location, delivery_location
                ]
                total_distance = distance_to_pickup + pickup_to_delivery
                score = total_distance * (
                    1 + state["load"] / self.instance.vehicle_capacity * 0.5
                )
                alternatives.append(
                    {
                        "vehicle_id": vehicle_id,
                        "score": float(score),
                        "distance": float(total_distance),
                        "load_after": state["load"] + demand,
                    }
                )

        alternatives.sort(key=lambda option: option["score"])
        if alternatives:
            best = alternatives[0]
            best_vehicle = best["vehicle_id"]
            best_state = vehicle_states[best_vehicle]
            decision = {
                "action": "assign_request",
                "request_id": request_id,
                "vehicle_id": best_vehicle,
                "confidence": max(0.5, 1.0 - (len(alternatives) - 1) * 0.1),
            }
            reasoning = [
                template.format(
                    request_id=request_id,
                    vehicle_id=best_vehicle,
                    current_load=best_state["load"],
                    capacity=self.instance.vehicle_capacity,
                    distance=self.instance.distances[
                        best_state["current_position"], pickup_location
                    ],
                    alternatives_count=len(alternatives) - 1,
                )
                for template in self.explanation_templates["route_assignment"]
            ]
            distance_factor = self.instance.distances[
                best_state["current_position"], pickup_location
            ]
            load_factor = best_state["load"] / self.instance.vehicle_capacity
            capacity_factor = (
                best_state["load"] + demand
            ) / self.instance.vehicle_capacity
            key_factors = {
                "distance_proximity": float(1 / (1 + distance_factor)),
                "load_efficiency": float(1 - load_factor),
                "capacity_utilization": float(capacity_factor),
                "feasible_vehicle_share": len(alternatives) / max(1, len(vehicle_states)),
            }
            confidence = decision["confidence"]
        else:
            decision = {
                "action": "no_assignment",
                "reason": "no_suitable_vehicle",
                "confidence": 0.8,
            }
            reasoning = [
                f"Request {request_id} was not assigned.",
                f"Request demand is {demand}.",
                "No listed vehicle has sufficient remaining capacity.",
            ]
            key_factors = {"capacity_constraint": 1.0}
            confidence = decision["confidence"]

        explanation = DecisionExplanation(
            decision_id=decision_id,
            decision_type="route_assignment",
            reasoning=reasoning,
            key_factors=key_factors,
            alternatives=alternatives[:3],
            confidence=confidence,
            timestamp=time.time(),
        )
        return decision, explanation

    def _adjust_priority_with_explanation(
        self, context: Dict[str, Any], decision_id: str
    ) -> Tuple[Dict[str, Any], DecisionExplanation]:
        request_id = context["request_id"]
        old_priority = context["current_priority"]
        urgency = context.get("urgency", 0.5)
        customer_importance = context.get("customer_importance", 0.5)
        time_pressure = context.get("time_pressure", 0.5)
        priority_score = urgency * 0.4 + customer_importance * 0.3 + time_pressure * 0.3

        if priority_score > 0.7:
            new_priority = "high"
            reason = "The weighted urgency score is high."
            schedule_impact = "Expedited processing is indicated by this rule."
        elif priority_score > 0.4:
            new_priority = "medium"
            reason = "The weighted urgency score is in the intermediate range."
            schedule_impact = "The standard processing category is retained."
        else:
            new_priority = "low"
            reason = "The weighted urgency score is low."
            schedule_impact = "The rule allows flexible scheduling."

        decision = {
            "action": "priority_adjustment",
            "request_id": request_id,
            "old_priority": old_priority,
            "new_priority": new_priority,
            "confidence": 0.8,
        }
        reasoning = [
            template.format(
                request_id=request_id,
                old_priority=old_priority,
                new_priority=new_priority,
                adjustment_reason=reason,
                schedule_impact=schedule_impact,
            )
            for template in self.explanation_templates["priority_adjustment"]
        ]
        key_factors = {
            "urgency": urgency,
            "customer_importance": customer_importance,
            "time_pressure": time_pressure,
            "weighted_priority_score": priority_score,
        }
        explanation = DecisionExplanation(
            decision_id=decision_id,
            decision_type="priority_adjustment",
            reasoning=reasoning,
            key_factors=key_factors,
            alternatives=[],
            confidence=0.8,
            timestamp=time.time(),
        )
        return decision, explanation

    def _optimize_route_with_explanation(
        self, context: Dict[str, Any], decision_id: str
    ) -> Tuple[Dict[str, Any], DecisionExplanation]:
        vehicle_id = context["vehicle_id"]
        current_route = list(context["current_route"])
        if not self._precedence_is_respected(current_route):
            raise ValueError("The input route must respect pickup-before-delivery precedence.")

        current_length = self.calculate_route_distance(current_route)
        improved_route = current_route.copy()
        best_length = current_length

        # Reversing internal segments preserves fixed route endpoints.
        for i in range(1, len(current_route) - 2):
            for j in range(i + 1, len(current_route) - 1):
                candidate = (
                    current_route[:i]
                    + current_route[i:j][::-1]
                    + current_route[j:]
                )
                if not self._precedence_is_respected(candidate):
                    continue
                candidate_length = self.calculate_route_distance(candidate)
                if candidate_length < best_length:
                    improved_route = candidate
                    best_length = candidate_length

        potential_savings = current_length - best_length
        if potential_savings > 0.1:
            decision = {
                "action": "route_optimization",
                "vehicle_id": vehicle_id,
                "old_route": current_route,
                "new_route": improved_route,
                "distance_saved": potential_savings,
                "confidence": 0.9,
            }
            reasoning = [
                template.format(
                    vehicle_id=vehicle_id,
                    current_length=current_length,
                    potential_savings=potential_savings,
                    method="precedence-aware 2-opt local search",
                )
                for template in self.explanation_templates["route_optimization"]
            ]
            key_factors = {
                "route_size": len(current_route),
                "relative_distance_reduction": potential_savings / current_length
                if current_length else 0.0,
                "candidate_feasibility": 1.0,
                "boundary-pair checks": len(current_route) ** 2,
            }
            confidence = 0.9
        else:
            decision = {
                "action": "no_optimization",
                "vehicle_id": vehicle_id,
                "reason": "no_improvement_above_threshold",
                "confidence": 0.8,
            }
            reasoning = [
                f"No accepted 2-opt improvement was found for vehicle {vehicle_id}.",
                f"Current route length: {current_length:.1f} distance units.",
                "The search result is local and does not establish global optimality.",
            ]
            key_factors = {
                "route_size": len(current_route),
                "accepted_distance_reduction": 0.0,
            }
            confidence = 0.8

        explanation = DecisionExplanation(
            decision_id=decision_id,
            decision_type="route_optimization",
            reasoning=reasoning,
            key_factors=key_factors,
            alternatives=[],
            confidence=confidence,
            timestamp=time.time(),
        )
        return decision, explanation


print("Explainable AI decision rules are defined.")

In [ ]:
class HumanOperator:
    """Simulated operator who evaluates explanations and may intervene."""

    def __init__(self, operator_id: str, expertise_level: float = 0.7):
        if not 0 <= expertise_level <= 1:
            raise ValueError("Expertise level must be between 0 and 1.")
        self.operator_id = operator_id
        self.expertise_level = expertise_level
        self.trust_metrics = TrustMetrics(
            human_id=operator_id,
            trust_score=0.5,
            competence_trust=0.5,
            integrity_trust=0.6,
            benevolence_trust=0.5,
            interaction_count=0,
            satisfaction_history=[],
            trust_history=[0.5],
        )
        self.feedback_history: List[HumanFeedback] = []
        self.intervention_history: List[Dict[str, Any]] = []
        self.preferences = {
            "detail_preference": 0.7,
            "autonomy_preference": 0.6,
            "intervention_threshold": 0.3,
        }

    def evaluate_explanation(
        self, explanation: DecisionExplanation
    ) -> Dict[str, Any]:
        """Calculate a transparent simulated evaluation from explanation features."""
        reasoning_quality = min(1.0, len(explanation.reasoning) / 5.0)
        factor_transparency = min(1.0, len(explanation.key_factors) / 4.0)
        alternative_awareness = min(1.0, len(explanation.alternatives) / 3.0)
        explanation_score = (
            reasoning_quality * 0.4
            + factor_transparency * 0.3
            + alternative_awareness * 0.3
        )
        if self.expertise_level > 0.7:
            explanation_score *= 1 + 0.1 * (self.expertise_level - 0.7)
        explanation_score = min(1.0, explanation_score)
        trust_impact = (explanation_score - 0.5) * 0.1

        return {
            "explanation_quality": explanation_score,
            "satisfaction": explanation_score,
            "trust_impact": trust_impact,
            "understanding_level": min(
                1.0, explanation_score * (1 + self.expertise_level * 0.2)
            ),
            "feedback_needed": explanation_score < 0.6
            or explanation.confidence < 0.7,
        }

    def provide_feedback(
        self,
        decision_id: str,
        evaluation: Dict[str, Any],
        comments: str = "",
    ) -> HumanFeedback:
        """Record feedback and update the operator's simulated trust score."""
        feedback = HumanFeedback(
            feedback_id=f"feedback_{len(self.feedback_history)}_{time.time_ns()}",
            decision_id=decision_id,
            human_id=self.operator_id,
            rating=max(1, min(5, int(evaluation["satisfaction"] * 5))),
            comments=comments,
            corrections={},
            trust_impact=evaluation["trust_impact"],
            timestamp=time.time(),
        )
        self.feedback_history.append(feedback)
        self.trust_metrics.trust_score = min(
            1.0,
            max(0.0, self.trust_metrics.trust_score + evaluation["trust_impact"]),
        )
        self.trust_metrics.interaction_count += 1
        self.trust_metrics.satisfaction_history.append(evaluation["satisfaction"])
        self.trust_metrics.trust_history.append(self.trust_metrics.trust_score)
        return feedback

    def should_intervene(
        self, decision: Dict[str, Any], explanation: DecisionExplanation
    ) -> bool:
        """Sample the intervention rule using confidence, trust, and complexity."""
        confidence_factor = 1.0 - explanation.confidence
        trust_factor = 1.0 - self.trust_metrics.trust_score
        complexity_factor = min(1.0, len(explanation.reasoning) / 10.0)
        probability = (
            confidence_factor * 0.4
            + trust_factor * 0.3
            + complexity_factor * 0.2
            + (1.0 - self.preferences["autonomy_preference"]) * 0.1
        )
        if self.expertise_level > 0.8:
            probability *= 1.2
        probability = min(1.0, max(0.0, probability))
        return random.random() < probability

    def make_intervention(
        self, decision: Dict[str, Any], explanation: DecisionExplanation
    ) -> Dict[str, Any]:
        """Record an intervention, selecting the second-ranked feasible vehicle when available."""
        intervention = {
            "intervention_id": f"intervention_{len(self.intervention_history)}_{time.time_ns()}",
            "human_id": self.operator_id,
            "original_decision": decision,
            "original_explanation": explanation,
            "intervention_type": "override",
            "modified_decision": None,
            "reasoning": (
                f"Operator {self.operator_id} reviewed the decision at "
                f"expertise level {self.expertise_level:.1f}."
            ),
            "timestamp": time.time(),
        }
        if (
            decision.get("action") == "assign_request"
            and len(explanation.alternatives) > 1
        ):
            alternative_vehicle = explanation.alternatives[1]["vehicle_id"]
            intervention["modified_decision"] = decision.copy()
            intervention["modified_decision"]["vehicle_id"] = alternative_vehicle
            intervention["reasoning"] += (
                f" The second-ranked feasible vehicle, {alternative_vehicle}, "
                "was selected by the demonstration rule."
            )
        self.intervention_history.append(intervention)
        return intervention


print("Human operator behavior is defined.")

In [ ]:
class HumanAIPartnership:
    """Coordinate the decision rules, simulated operators, and session metrics."""

    def __init__(self, instance: VRPPDInstance):
        self.instance = instance
        self.ai_system = ExplainableAI(instance)
        self.human_operators: Dict[str, HumanOperator] = {}
        self.partnership_history: List[Dict[str, Any]] = []
        self.performance_metrics = {
            "total_decisions": 0,
            "human_interventions": 0,
            "intervention_rate": 0.0,
            "average_trust": 0.0,
            "decision_quality_score": 0.0,
            "partnership_satisfaction": 0.0,
        }

    def add_human_operator(
        self, operator_id: str, expertise_level: float = 0.7
    ) -> None:
        self.human_operators[operator_id] = HumanOperator(
            operator_id, expertise_level
        )

    def make_collaborative_decision(
        self,
        decision_type: str,
        context: Dict[str, Any],
        primary_operator: Optional[str] = None,
    ) -> Dict[str, Any]:
        ai_decision, explanation = self.ai_system.make_decision_with_explanation(
            decision_type, context
        )
        if primary_operator is None and self.human_operators:
            primary_operator = next(iter(self.human_operators))

        result = {
            "decision_id": explanation.decision_id,
            "decision_type": decision_type,
            "ai_decision": ai_decision,
            "explanation": explanation,
            "final_decision": ai_decision,
            "human_intervention": False,
            "operator_feedback": None,
            "partnership_outcome": "ai_autonomous",
            "timestamp": time.time(),
        }

        if primary_operator in self.human_operators:
            operator = self.human_operators[primary_operator]
            evaluation = operator.evaluate_explanation(explanation)
            if operator.should_intervene(ai_decision, explanation):
                intervention = operator.make_intervention(ai_decision, explanation)
                result["human_intervention"] = True
                result["intervention"] = intervention
                result["final_decision"] = (
                    intervention["modified_decision"] or ai_decision
                )
                result["partnership_outcome"] = "human_intervention"
                self.performance_metrics["human_interventions"] += 1

            feedback = operator.provide_feedback(
                explanation.decision_id,
                evaluation,
                "Simulated evaluation of explanation quality and understanding.",
            )
            result["operator_feedback"] = feedback
            result["evaluation"] = evaluation
            result["trust_after_feedback"] = operator.trust_metrics.trust_score

        self.performance_metrics["total_decisions"] += 1
        self.performance_metrics["intervention_rate"] = (
            self.performance_metrics["human_interventions"]
            / self.performance_metrics["total_decisions"]
        )
        if self.human_operators:
            self.performance_metrics["average_trust"] = float(
                np.mean(
                    [
                        operator.trust_metrics.trust_score
                        for operator in self.human_operators.values()
                    ]
                )
            )
        self.partnership_history.append(result)
        return result

    def simulate_partnership_session(
        self, num_decisions: int = 50
    ) -> List[Dict[str, Any]]:
        """Run a sequence of simulated assignment, priority, and route decisions."""
        print(f"Starting a human–AI partnership session with {num_decisions} decisions.")
        session_results = []
        vehicle_states = {
            vehicle_id: {
                "current_position": self.instance.depot_start,
                "load": 0,
                "route": [self.instance.depot_start, self.instance.depot_end],
            }
            for vehicle_id in self.instance.vehicles
        }
        available_pairs = list(range(self.instance.num_pairs))

        for decision_num in range(num_decisions):
            possible_types = ["priority_adjustment", "route_optimization"]
            if available_pairs:
                possible_types.append("route_assignment")
            decision_type = random.choice(possible_types)

            if decision_type == "route_assignment":
                pair_index = random.choice(available_pairs)
                available_pairs.remove(pair_index)
                pickup = self.instance.pickups[pair_index]
                delivery = self.instance.deliveries[pair_index]
                context = {
                    "request_id": f"request_{pair_index + 1}",
                    "pickup_location": pickup,
                    "delivery_location": delivery,
                    "demand": self.instance.demands[pair_index],
                    "vehicle_states": vehicle_states,
                }
            elif decision_type == "priority_adjustment":
                context = {
                    "request_id": f"request_{decision_num + 1}",
                    "current_priority": random.choice(["low", "medium", "high"]),
                    "urgency": random.uniform(0, 1),
                    "customer_importance": random.uniform(0, 1),
                    "time_pressure": random.uniform(0, 1),
                }
            else:
                vehicle_id = random.choice(self.instance.vehicles)
                context = {
                    "vehicle_id": vehicle_id,
                    "current_route": vehicle_states[vehicle_id]["route"],
                }

            result = self.make_collaborative_decision(decision_type, context)
            session_results.append(result)

            if (
                decision_type == "route_assignment"
                and result["final_decision"].get("action") == "assign_request"
            ):
                vehicle_id = result["final_decision"]["vehicle_id"]
                state = vehicle_states[vehicle_id]
                state["route"] = state["route"][:-1] + [
                    context["pickup_location"],
                    context["delivery_location"],
                    self.instance.depot_end,
                ]
                # Each generated request is completed before the next assignment.
                state["load"] = 0
                state["current_position"] = self.instance.depot_start
            elif (
                decision_type == "route_optimization"
                and result["final_decision"].get("action") == "route_optimization"
            ):
                vehicle_id = result["final_decision"]["vehicle_id"]
                vehicle_states[vehicle_id]["route"] = result["final_decision"]["new_route"]

            if (decision_num + 1) % 10 == 0:
                print(
                    f"Decision {decision_num + 1}: {result['partnership_outcome']}; "
                    f"mean trust {self.performance_metrics['average_trust']:.2f}; "
                    f"intervention rate {self.performance_metrics['intervention_rate']:.2f}"
                )

        self._calculate_partnership_metrics()
        print("Partnership session completed.")
        return session_results

    def _calculate_partnership_metrics(self) -> None:
        """Summarize simulated explanation, trust, and intervention measures."""
        if not self.partnership_history:
            return
        autonomous = [
            result for result in self.partnership_history
            if not result["human_intervention"]
        ]
        intervention_results = [
            result for result in self.partnership_history
            if result["human_intervention"]
        ]
        qualities = [
            result["evaluation"]["explanation_quality"]
            for result in self.partnership_history
            if "evaluation" in result
        ]
        average_quality = float(np.mean(qualities)) if qualities else 0.0
        collaboration_fraction = len(intervention_results) / len(self.partnership_history)
        collaboration_adjustment = 0.1 * collaboration_fraction
        satisfaction = (
            self.performance_metrics["average_trust"] + average_quality
        ) / 2
        self.performance_metrics.update(
            {
                "decision_quality_score": (average_quality + collaboration_adjustment) * 100,
                "partnership_satisfaction": satisfaction,
                "avg_explanation_quality": average_quality,
                "ai_autonomous_rate": len(autonomous) / len(self.partnership_history),
                "collaboration_adjustment": collaboration_adjustment,
            }
        )


print("Human–AI partnership coordination is defined.")

In [ ]:
partnership = HumanAIPartnership(partnership_instance)
partnership.add_human_operator("operator_expert", expertise_level=0.9)
partnership.add_human_operator("operator_intermediate", expertise_level=0.6)
partnership.add_human_operator("operator_novice", expertise_level=0.3)

print("Human–AI partnership system initialized:")
print(f"- Simulated operators: {len(partnership.human_operators)}")
print("- Expertise levels: 0.9, 0.6, and 0.3")
print(f"- Pickup–delivery pairs available: {partnership_instance.num_pairs}")
print("- Session length: 50 simulated decisions")

session_results = partnership.simulate_partnership_session(num_decisions=50)
print("\nThe simulation completed. Its metrics are defined by the rules in this notebook.")

In [ ]:
def analyze_partnership_results(
    results: List[Dict[str, Any]], partnership: HumanAIPartnership
) -> Dict[str, Any]:
    """Calculate descriptive metrics from a completed simulation."""
    if not results:
        return {"error": "There are no results to analyze."}

    total_decisions = len(results)
    human_interventions = sum(
        result["human_intervention"] for result in results
    )
    intervention_rate = human_interventions / total_decisions
    decision_types: Dict[str, Dict[str, int]] = {}
    for result in results:
        decision_type = result["decision_type"]
        decision_types.setdefault(decision_type, {"total": 0, "interventions": 0})
        decision_types[decision_type]["total"] += 1
        decision_types[decision_type]["interventions"] += int(
            result["human_intervention"]
        )

    qualities = [
        result["evaluation"]["explanation_quality"]
        for result in results
        if "evaluation" in result
    ]
    satisfaction = [
        result["evaluation"]["satisfaction"]
        for result in results
        if "evaluation" in result
    ]
    trust_development = []
    for operator_id, operator in partnership.human_operators.items():
        trust = operator.trust_metrics
        trust_development.append(
            {
                "operator_id": operator_id,
                "initial_trust": trust.trust_history[0],
                "final_trust": trust.trust_score,
                "trust_improvement": trust.trust_score - trust.trust_history[0],
                "interactions": trust.interaction_count,
                "avg_satisfaction": float(np.mean(trust.satisfaction_history))
                if trust.satisfaction_history else 0.0,
            }
        )

    average_quality = float(np.mean(qualities)) if qualities else 0.0
    average_trust = float(
        np.mean([item["final_trust"] for item in trust_development])
    ) if trust_development else 0.0
    average_satisfaction = float(np.mean(satisfaction)) if satisfaction else 0.0
    return {
        "total_decisions": total_decisions,
        "human_interventions": human_interventions,
        "intervention_rate": intervention_rate,
        "decision_types": decision_types,
        "avg_explanation_quality": average_quality,
        "avg_trust_score": average_trust,
        "avg_satisfaction": average_satisfaction,
        "trust_development": trust_development,
        "partnership_effectiveness": (1 - intervention_rate) * average_quality,
    }


analysis = analyze_partnership_results(session_results, partnership)
print("=== SIMULATION SUMMARY ===")
print(f"Total decisions: {analysis['total_decisions']}")
print(f"Human interventions: {analysis['human_interventions']}")
print(f"Intervention rate: {analysis['intervention_rate']:.2%}")
print(f"Mean explanation score: {analysis['avg_explanation_quality']:.2f}")
print(f"Mean final trust score: {analysis['avg_trust_score']:.2f}")
print(f"Mean satisfaction score: {analysis['avg_satisfaction']:.2f}")
print(f"Defined partnership-effectiveness index: {analysis['partnership_effectiveness']:.2f}")

print("\nTrust history by operator:")
for trust_info in analysis["trust_development"]:
    print(
        f"- {trust_info['operator_id']}: "
        f"{trust_info['initial_trust']:.2f} to {trust_info['final_trust']:.2f}; "
        f"{trust_info['interactions']} interactions"
    )

print("\nIntervention rate by decision type:")
for decision_type, stats in analysis["decision_types"].items():
    rate = stats["interventions"] / stats["total"] if stats["total"] else 0.0
    print(
        f"- {decision_type}: {rate:.2%} "
        f"({stats['interventions']}/{stats['total']})"
    )

In [ ]:
def visualize_partnership_results(
    results: List[Dict[str, Any]],
    analysis: Dict[str, Any],
    partnership: HumanAIPartnership,
):
    """Plot simulated trust, interventions, evaluation scores, and summary measures."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    ax1, ax2, ax3, ax4 = axes.flat
    fig.suptitle(
        "VRPPD Human–AI Decision-Support Simulation",
        fontsize=16,
        fontweight="bold",
    )

    ax1.set_title("Recorded Trust by Interaction")
    for operator_id, operator in partnership.human_operators.items():
        trust_history = operator.trust_metrics.trust_history
        ax1.plot(
            range(len(trust_history)),
            trust_history,
            marker="o",
            linewidth=2,
            label=operator_id,
        )
    ax1.set_xlabel("Interaction number (including initial value)")
    ax1.set_ylabel("Simulated trust score")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)
    ax1.legend(title="Operator")

    ax2.set_title("Decision Outcomes")
    autonomous = [
        index for index, result in enumerate(results)
        if not result["human_intervention"]
    ]
    interventions = [
        index for index, result in enumerate(results)
        if result["human_intervention"]
    ]
    ax2.scatter(
        autonomous, [1] * len(autonomous), alpha=0.7,
        color="seagreen", label="AI decision retained",
    )
    ax2.scatter(
        interventions, [0] * len(interventions), alpha=0.8,
        color="indianred", label="Human intervention recorded",
    )
    ax2.set_xlabel("Decision number")
    ax2.set_ylabel("Outcome category")
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(["Intervention", "AI retained"])
    ax2.set_ylim(-0.5, 1.5)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.annotate(
        f"Intervention rate: {analysis['intervention_rate']:.1%}",
        xy=(0.02, 0.98),
        xycoords="axes fraction",
        va="top",
        bbox={"boxstyle": "round", "facecolor": "aliceblue", "alpha": 0.8},
    )

    ax3.set_title("Explanation Score and Satisfaction")
    qualities = []
    satisfaction = []
    for result in results:
        if "evaluation" in result:
            qualities.append(result["evaluation"]["explanation_quality"])
            satisfaction.append(result["evaluation"]["satisfaction"])
    if qualities:
        ax3.scatter(qualities, satisfaction, alpha=0.7, s=40, color="slateblue")
        ax3.set_xlim(0, 1)
        ax3.set_ylim(0, 1)
    ax3.set_xlabel("Rule-based explanation score")
    ax3.set_ylabel("Rule-based satisfaction score")
    ax3.grid(True, alpha=0.3)
    ax3.text(
        0.03, 0.97,
        "Satisfaction is defined from the same score in this demonstration.",
        transform=ax3.transAxes,
        va="top",
        fontsize=9,
        wrap=True,
    )

    ax4.set_title("Descriptive Simulation Measures")
    metrics = {
        "Final trust": analysis["avg_trust_score"],
        "Explanation score": analysis["avg_explanation_quality"],
        "Satisfaction": analysis["avg_satisfaction"],
        "Effectiveness index": analysis["partnership_effectiveness"],
    }
    categories = list(metrics)
    values = list(metrics.values())
    bars = ax4.barh(
        categories,
        values,
        color=["steelblue", "mediumseagreen", "goldenrod", "indianred"],
        edgecolor="black",
    )
    ax4.set_xlim(0, 1)
    ax4.set_xlabel("Score (0–1)")
    ax4.grid(True, axis="x", alpha=0.3)
    for bar, value in zip(bars, values):
        ax4.text(
            min(value + 0.02, 0.94),
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            va="center",
            fontweight="bold",
        )

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_partnership_results(session_results, analysis, partnership)

In [ ]:
def demonstrate_explanation_system(partnership: HumanAIPartnership) -> None:
    """Show one assignment decision, its explanation, and an operator evaluation."""
    print("=== EXPLAINABLE DECISION DEMONSTRATION ===")
    context = {
        "request_id": "DEMO_001",
        "pickup_location": partnership.instance.pickups[2],
        "delivery_location": partnership.instance.deliveries[2],
        "demand": partnership.instance.demands[2],
        "vehicle_states": {
            0: {"current_position": 0, "load": 3},
            1: {"current_position": 5, "load": 1},
            2: {"current_position": 2, "load": 5},
        },
    }
    decision, explanation = partnership.ai_system.make_decision_with_explanation(
        "route_assignment", context
    )
    print(f"Decision ID: {explanation.decision_id}")
    print(f"Decision type: {explanation.decision_type}")
    print(f"Decision: {decision}")
    print(f"Rule confidence: {explanation.confidence:.2f}")

    print("\nReasoning:")
    for step_number, step in enumerate(explanation.reasoning, start=1):
        print(f"{step_number}. {step}")

    print("\nFactors used by the demonstration:")
    for factor, value in explanation.key_factors.items():
        print(f"- {factor}: {value:.3f}")

    print("\nFeasible alternatives (ranked by the greedy score):")
    for rank, alternative in enumerate(explanation.alternatives, start=1):
        print(
            f"{rank}. Vehicle {alternative['vehicle_id']}: "
            f"score {alternative['score']:.2f}, "
            f"distance {alternative['distance']:.1f}, "
            f"load after assignment {alternative['load_after']}"
        )

    operator = partnership.human_operators["operator_expert"]
    evaluation = operator.evaluate_explanation(explanation)
    print("\nSimulated operator evaluation:")
    print(f"Explanation score: {evaluation['explanation_quality']:.2f}")
    print(f"Understanding score: {evaluation['understanding_level']:.2f}")
    print(f"Satisfaction score: {evaluation['satisfaction']:.2f}")
    print(f"Trust impact: {evaluation['trust_impact']:+.3f}")
    print(f"Feedback indicated: {'Yes' if evaluation['feedback_needed'] else 'No'}")

    feedback = operator.provide_feedback(
        explanation.decision_id,
        evaluation,
        "Simulated feedback on the explanation structure.",
    )
    print("\nRecorded simulated feedback:")
    print(f"Rating: {feedback.rating}/5")
    print(f"Trust impact: {feedback.trust_impact:+.3f}")
    print(f"Updated simulated trust: {operator.trust_metrics.trust_score:.3f}")


demonstrate_explanation_system(partnership)

## Role, strengths, and limitations

### Role of human–AI collaboration
This layer makes heuristic decisions easier to inspect and gives a human operator a simulated opportunity to review them. It is useful when operational decisions need explanation, oversight, or a recorded feedback process. It complements rather than replaces a routing algorithm: the assignment and 2-opt routines in this notebook remain simple heuristics.

### Strengths demonstrated
- The assignment rule exposes its score, capacity screen, and ranked feasible alternatives.
- The operator model represents feedback, trust updates, and a confidence-influenced intervention probability.
- The simulation records outcomes that can be inspected by decision type and operator.
- The route search checks pickup-before-delivery precedence for the represented request pairs.

### Limitations
- Explanation quality, satisfaction, confidence, and trust are generated by hand-designed rules; they are not validated psychological measurements.
- Feedback updates the simulated trust score only. It does not train the assignment rule or adapt the AI policy.
- A human intervention selects a second-ranked feasible vehicle when one is available. It does not apply independent domain knowledge or verify a real operational plan.
- The route-assignment check considers current load plus pickup demand, but the session treats each assigned request as completed before the next assignment. It is not a vehicle schedule with simultaneous onboard loads, time windows, service times, or route-duration constraints.
- The 2-opt search is local. Its accepted moves preserve represented pickup-before-delivery precedence, but the demonstration does not model every operational constraint.
- Simulation metrics do not establish that collaboration improves decisions relative to an independently evaluated baseline.

### Appropriate use
Use this example to study transparent decision rules, human review, feedback bookkeeping, and descriptive simulation analysis. A deployment or empirical study would require operationally complete constraints, validated measures, representative data, and a defined process for assessing decision quality.

In [ ]:
def final_summary(
    analysis: Dict[str, Any], partnership: HumanAIPartnership
) -> None:
    """Print a data-dependent summary and clarify what the metrics represent."""
    print("\n" + "=" * 72)
    print("VRPPD HUMAN–AI DECISION-SUPPORT DEMONSTRATION")
    print("=" * 72)

    print("\nDecision-support components:")
    print("- Greedy capacity-screened assignment with ranked alternatives")
    print("- Weighted priority rule and precedence-aware 2-opt local search")
    print("- Explanation records, simulated operator feedback, and trust updates")
    print("- Session-level descriptive analysis and visualizations")

    print("\nMetrics from this run:")
    print(f"- Decisions: {analysis['total_decisions']}")
    print(f"- Recorded interventions: {analysis['human_interventions']}")
    print(f"- Intervention rate: {analysis['intervention_rate']:.2%}")
    print(f"- Mean explanation score: {analysis['avg_explanation_quality']:.2f}")
    print(f"- Mean final trust score: {analysis['avg_trust_score']:.2f}")
    print(f"- Mean satisfaction score: {analysis['avg_satisfaction']:.2f}")
    print(f"- Defined effectiveness index: {analysis['partnership_effectiveness']:.2f}")

    print("\nOperators:")
    for item in analysis["trust_development"]:
        print(
            f"- {item['operator_id']}: trust {item['initial_trust']:.2f} to "
            f"{item['final_trust']:.2f} across {item['interactions']} interactions"
        )

    print("\nInterpretation:")
    print(
        "These values summarize the simulation's explicit scoring and update rules. "
        "They are not measured human outcomes or evidence of improved routing quality."
    )
    print(
        "The notebook demonstrates a decision-support layer; it does not solve the "
        "full vehicle routing problem with pickup and delivery."
    )

    print("\nPotential extensions within this modeling scope:")
    print("- Represent service times, time windows, and simultaneous vehicle loads.")
    print("- Define and validate human evaluation measures before interpreting them empirically.")
    print("- Evaluate the same decision rules on larger, documented instances.")
    print("=" * 72)


final_summary(analysis, partnership)